# Instruction holdout evaluation
One T4 + Internet. Separate from the chatbot serving session. Add the exact holdout-evaluation ZIP dataset; no checkpoint Input or secrets needed. Base weights may download once. Run cells in order: authenticate, environment, validation smoke, 116 holdout. No training.


In [ ]:
from pathlib import Path
import hashlib, json, os, stat, zipfile, subprocess, sys, time
from datetime import datetime, timezone
os.environ.update(CUDA_VISIBLE_DEVICES='0', USE_TF='0', USE_FLAX='0', PYTHONDONTWRITEBYTECODE='1', TOKENIZERS_PARALLELISM='false')
EXPECTED = 'c2ceeec9c53a857b6e5b8304a96ea0ba6b7fc0ce961fdf72d937e34fe50ebb41'
EXPECTED_TRANSPORT = '6cbf631f509db37cc6607cf6f33600b4d10b4e20350c3d93fe91bfed995e0c58'
matches = [p for p in Path('/kaggle/input').rglob('holdout-evaluation.zip') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED]
PACKAGE = Path('/kaggle/working/holdout-package-' + EXPECTED[:12])
if matches:
    assert len(matches) == 1, 'Duplicate matching ZIP inputs.'
    with zipfile.ZipFile(matches[0]) as z:
        names = z.namelist()
        assert len(names) == len(set(names))
        transport = z.read('bundle-manifest.json')
        assert hashlib.sha256(transport).hexdigest() == EXPECTED_TRANSPORT
        manifest = json.loads(transport)
        assert set(names) == set(manifest) | {'bundle-manifest.json'}
        assert all(not stat.S_ISLNK(i.external_attr >> 16) for i in z.infolist())
        payload = {name: z.read(name) for name in manifest}
else:
    manifests = [p for p in Path('/kaggle/input').rglob('bundle-manifest.json') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_TRANSPORT]
    assert len(manifests) == 1, 'Add the exact runner-dev ZIP or its extracted Kaggle Dataset.'
    source = manifests[0].parent.resolve()
    manifest = json.loads(manifests[0].read_text())
    payload = {}
    for name in manifest:
        p = (source / name).resolve()
        assert p.is_relative_to(source) and not (source / name).is_symlink()
        payload[name] = p.read_bytes()
for name, raw in payload.items():
    target = (PACKAGE / name).resolve()
    assert target.is_relative_to(PACKAGE.resolve()) and not Path(name).is_absolute()
    assert chr(92) not in name and ':' not in name
    assert len(raw) == manifest[name]['bytes'] and hashlib.sha256(raw).hexdigest() == manifest[name]['sha256']
for name, raw in payload.items():
    target = PACKAGE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == manifest[name]['sha256']
    else:
        with target.open('xb') as f: f.write(raw)
PYTHON = Path('/kaggle/working/holdout-env/bin/python')
print('Holdout package authenticated. No training. Qwen not loaded yet.')


In [ ]:
ENV = Path('/kaggle/working/holdout-env')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(ENV)], check=True)
from importlib.metadata import version
torch_before = version('torch')
constraint = ENV / 'torch-constraint.txt'
constraint.write_text('torch==' + torch_before + '\n')
subprocess.run([sys.executable, '-m', 'pip', '--python', str(PYTHON), 'install', '--disable-pip-version-check', '--constraint', str(constraint), '-r', str(PACKAGE / 'scripts/comparison/requirements.txt')], check=True)
torch_after = subprocess.run([str(PYTHON), '-c', 'from importlib.metadata import version; print(version("torch"))'], check=True, capture_output=True, text=True).stdout.strip()
assert torch_before == torch_after, 'CUDA torch changed'
subprocess.run([str(PYTHON), '-B', '-m', 'scripts.comparison.holdout', '--package', str(PACKAGE), '--check-only'], cwd=PACKAGE, check=True)
print('Environment and 116 holdout token/mask checks passed. GPU evaluation not run yet.')


In [ ]:
def execute(mode):
    run = Path('/kaggle/working/holdout-runs') / (mode + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    run.parent.mkdir(parents=True, exist_ok=True)
    console = Path(str(run) + '-console.log')
    started = time.monotonic()
    with console.open('x', encoding='utf-8') as log:
        process = subprocess.Popen([str(PYTHON), '-B', '-m', 'scripts.comparison.holdout', '--package', str(PACKAGE), '--output', str(run), '--split', mode], cwd=PACKAGE, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                progress = json.loads((run / 'progress.json').read_text())
            except (OSError, json.JSONDecodeError):
                progress = dict(phase='STARTING', rows=0)
            print(str(int(time.monotonic()-started)) + 's | ' + json.dumps(progress), flush=True)
            time.sleep(15)
    audit = json.loads((run / 'audit.json').read_text()) if (run / 'audit.json').exists() else None
    archive = Path(str(run) + '.zip')
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(console, 'console.log')
        for p in sorted(run.rglob('*')):
            if p.is_file(): z.write(p, p.relative_to(run).as_posix())
    receipt = dict(archive=archive.name, sha256=hashlib.sha256(archive.read_bytes()).hexdigest(), size_bytes=archive.stat().st_size, process_exit_code=process.returncode, audit=audit)
    receipt_path = Path(str(run) + '-receipt.json')
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
    print(json.dumps(receipt, indent=2))
    print('ZIP:', archive, '\nReceipt:', receipt_path, '\nAudit:', run / 'audit.json')
    assert process.returncode == 0 and audit is not None, 'Run incomplete; the ZIP and receipt preserve available failure evidence.'
    return run, audit
SMOKE_RUN, SMOKE_AUDIT = execute('smoke')


In [ ]:
assert SMOKE_AUDIT['rows'] == 8 and SMOKE_AUDIT['failed_generations'] == 0, 'Resolve validation smoke failures before opening holdout.'
HOLDOUT_RUN, HOLDOUT_AUDIT = execute('holdout')
assert HOLDOUT_AUDIT['rows'] == 232
print('Figures:', HOLDOUT_RUN / 'figures')
print('Send holdout ZIP, receipt and audit.json. Do not select or retrain an adapter using these scores.')
